### Model downloadnd summary

In [ ]:
import os
target_dir_path = '/home/diego/Documents/Projects/tracked/eciyali5_SF_BIO'
os.chdir(target_dir_path)

In [ ]:
import os
from pathlib import Path
import cobra
import yaml
from cobra.io.dict import model_from_dict
from utils.data_utils import *

# Map desired dictionary keys to file names
model_files = {
    "exp": "ecPARe_pooled.yml",
    "nlim": "ecPARn_pooled.yml"
}

# Initialize the models dictionary
models = {}

#Iterate through keys and files
for phase, filename in model_files.items():
    model_path = f"{target_dir_path}/models/{filename}"
    print(f"Loading {phase} model from {model_path}...")

    # 3. Parse YAML safely
    with open(model_path, "r", encoding="utf-8") as f:
        model_dict = yaml.load(f, Loader=MatlabYamlSafeLoader)

    print(f"YAML successfully parsed for {phase}! Constructing COBRA model...")

    # 4. Instantiate and store in the dictionary
    models[phase] = model_from_dict(model_dict)

print("All models loaded successfully!")

In [ ]:
#summarise models.
for model in models.values():
    display(model)
    display(model.objective)

df_exp = analyze_boundary_reactions(models["exp"], show_pseudometabolites=False)

#exp and nlim are the same at this point; this is illustrating the stochiometric limitations of both models in export and import.
#the only difference between exp and nlim at this point are their NGAM and GAM reactions.
print("Import reactions:")
df_exp_in = df_exp[df_exp["Allows_Import"]].sort_values(by="Max_Import_Rate", ascending=False)
display(df_exp_in)

print("Export reactions:")
df_exp_out = df_exp[df_exp["Allows_Export"]].sort_values(by="Max_Export_Rate", ascending=False)
display(df_exp_out)

### Initialisation of constrained models; Stochiometric Prediction of O2 and CO2 fluxes; Initialisation of minimally-constrained models

In [ ]:
#IMPORTANT - REMOVED L ARABITINOL BECAUSE IT IS MINIMALLY PRODUCED AND IS NOT IN THE MODEL.
#NEED PAPER EVIDENCE FOR THIS THOUGH!
#figure out why model infeasible

F = 0.5 #enzyme fraction
sigma = 0.49 #average enzyme saturation factor - using sigmaFitter on glycerol

# Define directly by Reaction_ID: observed flux [mmol/gDCW/h, h^-1, or mg/gDCW for pool]
test_fluxes = {
    "BIO_exp_const": {
        "xBIOMASS": 0.45,                 # Biomass growth rate
        "1808": -10.7,                # Glycerol uptake
        "1687": 0.0,                  # Citrate export
        "EXC_OUT_isocitrate": 0.0,      # Isocitrate export
        "EXC_OUT_Dmannitol": 0.0,       # D-Mannitol export
        "EXC_OUT_erythritol": 0.0,      # Erythritol export
        "prot_pool_exchange": 0.202*F*sigma*1000,   # Protein pool bound
    },
    "BIO_nlim_const": {
        "xBIOMASS": 0.02,
        "1808": -4.5,
        "1687": 1.5,
        "EXC_OUT_isocitrate": 0.15,
        "EXC_OUT_Dmannitol": 0.0,       # Set > 0 for shake-flask hypoxia
        "EXC_OUT_erythritol": 0.0,      # Set > 0 for shake-flask hypoxia
        "prot_pool_exchange": 0.103*F*sigma*1000,
    },
    "SF_exp_const": {
            "xBIOMASS": 0.45,                 # Biomass growth rate
            "1808": -10.7,                # Glycerol uptake
            "1687": 0.0,                  # Citrate export
            "EXC_OUT_isocitrate": 0.0,      # Isocitrate export
            "EXC_OUT_Dmannitol": 0.1,       # D-Mannitol export
            "EXC_OUT_erythritol": 0.1,      # Erythritol export
            "prot_pool_exchange": 0.202*F*sigma*1000,   # Protein pool bound
        },
    "SF_nlim_const": {
        "xBIOMASS": 0.02,
        "1808": -4.5,
        "1687": 0.25,
        "EXC_OUT_isocitrate": 0.25,
        "EXC_OUT_Dmannitol": 0.4,       # Set > 0 for shake-flask hypoxia
        "EXC_OUT_erythritol": 0.4,      # Set > 0 for shake-flask hypoxia
        "prot_pool_exchange": 0.103*F*sigma*1000,
        }
}


# Copies the base template ("exp" or "nlim") into a new condition-specific model
for condition, fluxes in test_fluxes.items():
    base_phase = "exp" if "exp" in condition else "nlim"
    new_m = models[base_phase].copy()
    new_m.id = condition
    rows = []

    for rxn_id, val in fluxes.items():
        if rxn_id not in new_m.reactions:
            print(f"Reaction ID '{rxn_id}' not found in {condition}.")
            continue

        rxn = new_m.reactions.get_by_id(rxn_id)

        if rxn_id == "prot_pool_exchange":
            lb, ub = (val if val < 0 else -val), 0.0
        elif val == 0.0:
            lb, ub = 0.0, 0.0
        else:
            lb, ub = min(val * 0.9, val * 1.1), max(val * 0.9, val * 1.1)

        rows.append({
            "Reaction_ID": rxn.id,
            "Name": rxn.name,
            "Obs_Flux": val,
            "Old_LB": rxn.lower_bound,
            "New_LB": lb,
            "Old_UB": rxn.upper_bound,
            "New_UB": ub,
        })
        rxn.bounds = (float(lb), float(ub))

    print(f"\n--- {condition} CONSTRAINTS ---")
    display(pd.DataFrame(rows))

    models[condition] = new_m
    print(f"Stored model in models['{condition}'].")

In [ ]:
#Assess predicted O2 and CO2 flux.
for model in models.values():
    if model.id is not None:
        if str(model.id)[-6:]=="_const":
            print(f"\n--- Model: {model.id} ---")
            sol = model.optimize()
            o2 = sol.fluxes.get("1992", 0)   # Oxygen exchange
            co2 = sol.fluxes.get("1672", 0)  # CO2 exchange
            print(f"Predicted O2: {o2:.3f} | CO2: {co2:.3f} | RQ: {abs(co2/o2):.2f}")

In [ ]:
#Generate minimally constrained models.

conds = ["BIO_exp_const", "BIO_nlim_const", "SF_exp_const", "SF_nlim_const"]

# Extract predicted O2 limits from the _const runs
o2_limits = {k: models[f"{k}"].optimize().fluxes.get("1992", -1000.0) for k in conds}

# Build the 4 condition-anchored models
p_bounds = {"exp": -0.202 * F * sigma * 1000, "nlim": -0.103 * F * sigma * 1000}
gly_rates = {"exp": -10.7, "nlim": -4.5}
open_rxns = ["1687", "EXC_OUT_isocitrate", "EXC_OUT_Dmannitol", "EXC_OUT_erythritol", "1875"]

summary = []
for cond in conds:
    phase = "exp" if "exp" in cond else "nlim"
    m = models[phase].copy()
    m.id = cond[:-6] #remove _const suffix

    m.reactions.get_by_id("1808").bounds = (gly_rates[phase], gly_rates[phase])
    m.reactions.get_by_id("prot_pool_exchange").bounds = (p_bounds[phase], 0.0)
    m.reactions.get_by_id("xBIOMASS").bounds = (0.01, 1000.0)
    m.reactions.get_by_id("1992").bounds = (-1000.0 if "BIO" in cond else float(o2_limits[cond]), 0.0)

    for r_id in open_rxns:
        if r_id in m.reactions:
            m.reactions.get_by_id(r_id).bounds = (0.0, 1000.0)

    models[cond[:-6]] = m
    summary.append({
        "Model": cond[:-6],
        "1808 (Gly)": m.reactions.get_by_id("1808").bounds,
        "prot_pool": m.reactions.get_by_id("prot_pool_exchange").bounds,
        "1992 (O2)": m.reactions.get_by_id("1992").bounds,
        "1687 (Cit)": m.reactions.get_by_id("1687").bounds
    })

display(pd.DataFrame(summary))

oxygen is NEVER LIMITING IN BIOREACTOR DUE TO OXYGEN SENSOR ENSURING ALWAYS AT 30% NO MATTER WHAT.

Compare bioreactor oxygen consumption with predicted consumption from model carbon. Use this to generate uncertainty bounds on the model's oxygen consumption.

In [ ]:
#do the above here.

### Constrained models - proteome allocation

In [ ]:
#saturation might be broken - maximising always at 100%

pool_stats = []
for cond in ["BIO_exp_const", "BIO_nlim_const", "SF_exp_const", "SF_nlim_const"]:
    m = models[cond]
    sol = m.optimize()
    rxn = m.reactions.get_by_id("prot_pool_exchange")
    
    cap = abs(rxn.lower_bound)
    used = abs(sol.fluxes.get("prot_pool_exchange", 0.0))
    
    pool_stats.append({
        "Model": cond,
        "Used (mg/gDCW)": round(used, 2),
        "Capacity (mg/gDCW)": round(cap, 2),
        "Saturation (%)": round((used / cap) * 100, 2) if cap > 0 else 0.0,
        "Status": sol.status
    })

display(pd.DataFrame(pool_stats))

In [ ]:
#Proteome allocaiton for relevant conditions
#running proteome allocaiton does not require model to be 100% saturated.
print("--- PROTEOME ALLOCATION ---")
display(get_proteome_allocation(models["BIO_nlim_const"]).head(20))
display(get_proteome_allocation(models["SF_nlim_const"]).head(20))

### Constrained models - metabolic flux distribution analysis

In [ ]:
#do the above

### Minimally-constrained models; FSEOF and FCC for citrate productivity. How do they differ?

In [ ]:
#polyol export check for minimaly constrained models
#proves that polyol export is required under oxygen limitation to meet respiratory demand of biomass maintenance.
import pandas as pd

polyol_rxns = {
    "D-Mannitol": "EXC_OUT_Dmannitol",
    "Erythritol": "EXC_OUT_erythritol",
}

# Check both the anchored models and the fully constrained versions
target_models = ["BIO_nlim", "SF_nlim"]

records = []
for name in target_models:
    if name in models:
        m = models[name]
        sol = m.optimize()
        
        for polyol, r_id in polyol_rxns.items():
            if r_id in m.reactions:
                rxn = m.reactions.get_by_id(r_id)
                flux = sol.fluxes.get(r_id, 0.0) if sol.status == "optimal" else 0.0
                records.append({
                    "Model": name,
                    "Polyol": polyol,
                    "Reaction_ID": r_id,
                    "Lower_Bound": rxn.lower_bound,
                    "Upper_Bound": rxn.upper_bound,
                    "Active_Flux": round(flux, 4),
                    "Is_Exporting": flux > 1e-5
                })

display(pd.DataFrame(records))

,Model,Polyol,Reaction_ID,Lower_Bound,Upper_Bound,Active_Flux,Is_Exporting
0,BIO_nlim,D-Mannitol,EXC_OUT_Dmannitol,0.0,1000.0,0.0000,False
1,BIO_nlim,Erythritol,EXC_OUT_erythritol,0.0,1000.0,0.0000,False
2,SF_nlim,D-Mannitol,EXC_OUT_Dmannitol,0.0,1000.0,0.0000,False
3,SF_nlim,Erythritol,EXC_OUT_erythritol,0.0,1000.0,0.5495,True


In [ ]:
#FSEOF
#running FSEOF does not require model to be 100% saturated.

print("\n>>> RUNNING NLIM Bioreactor...")
BIO_nlim_rxns = run_fseof_debiaggi(
    model=models['BIO_nlim'],  
    prod_target_rxn="1687", #"EXC_OUT_m1640"
    bio_rxn="xBIOMASS",
    n_steps=16,
    filter_complex_rxns=True
)

print("\n>>> RUNNING NLIM Shake flask...")
SF_nlim_rxns = run_fseof_debiaggi(
    model=models['SF_nlim'],  
    prod_target_rxn="1687", #"EXC_OUT_m1640"
    bio_rxn="xBIOMASS",
    n_steps=16,
    filter_complex_rxns=True
)

In [ ]:
#FCC
#running FCC DOES!!! require model to be 100% saturated.
fcc_bio = calculate_fcc_kcat(models["BIO_nlim"], target_rxn="xBIOMASS", Qa=1.01, top_n=10)
fcc_sf = calculate_fcc_kcat(models["SF_nlim"], target_rxn="xBIOMASS", Qa=1.01, top_n=10)

print("\n--- TOP 10 FCC: BIOREACTOR vs SHAKE FLASK ---")
display(fcc_bio.head(10))
display(fcc_sf.head(10))